# 1. Data Loading and Initial Inspection

## 1.1 Importing Libraries

In [1]:
#importing libraries for data manipulation
import pandas as pd
import numpy as np

## 1.2 Loading the Dataset

In [2]:
#loading the dataset
df=pd.read_csv('../data/raw/Car details v3.csv')
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [3]:
df.shape

(8128, 13)

## 1.3 Removing Duplicate Records

In [4]:
#removing duplicates and copying the changed dataset
df_clean=df.drop_duplicates().copy()
df_clean.shape

(6926, 13)

## 1.4 Initial Missing Value Check

In [5]:
#missing values after removing duplicates
df_clean.isnull().sum()

name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
mileage          208
engine           208
max_power        205
torque           209
seats            208
dtype: int64

# 2. Mileage Cleaning

# 2.1 Exploring Mileage Values and Units

In [6]:
#exploring mileage
df_clean['mileage'].head(10)

0     23.4 kmpl
1    21.14 kmpl
2     17.7 kmpl
3     23.0 kmpl
4     16.1 kmpl
5    20.14 kmpl
6    17.3 km/kg
7     16.1 kmpl
8    23.59 kmpl
9     20.0 kmpl
Name: mileage, dtype: str

In [7]:
#checking how many values each unit uses
df_clean['mileage'].str.extract(r'(kmpl|km/kg)',expand=False).value_counts()

mileage
kmpl     6631
km/kg      87
Name: count, dtype: int64

In [8]:
#vehicles which are using km/kg as mileage unit
df_clean[df_clean['mileage'].str.contains('km/kg',na=False)].head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
6,Maruti Wagon R LXI DUO BSIII,2007,96000,175000,LPG,Individual,Manual,First Owner,17.3 km/kg,1061 CC,57.5 bhp,"7.8@ 4,500(kgm@ rpm)",5.0
35,Maruti Alto 800 CNG LXI Optional,2019,330000,10000,CNG,Individual,Manual,Second Owner,33.44 km/kg,796 CC,40.3 bhp,60Nm@ 3500rpm,4.0
90,Maruti Wagon R LXI LPG BSIV,2010,225000,44000,LPG,Dealer,Manual,First Owner,26.2 km/kg,998 CC,58.2 bhp,77Nm@ 3500rpm,5.0
203,Maruti Wagon R LXI CNG,2016,360000,50000,CNG,Individual,Manual,First Owner,26.6 km/kg,998 CC,58.16 bhp,77Nm@ 3500rpm,5.0
402,Maruti Eeco CNG 5 Seater AC,2020,409999,35000,CNG,Individual,Manual,First Owner,20.88 km/kg,1196 CC,61.7 bhp,85Nm@ 3000rpm,5.0


In [9]:
#fuel types which have km/kg as mileage unit
df_clean.loc[df_clean['mileage'].str.contains('km/kg',na=False),'fuel'].value_counts()

fuel
CNG    52
LPG    35
Name: count, dtype: int64

In [10]:
#checking whether CNG and LPG cars have kmpl
df_clean.loc[df_clean['fuel'].isin(['CNG','LPG']),'mileage'].str.extract(r'(kmpl|km/kg)',expand=False).value_counts()

mileage
km/kg    87
Name: count, dtype: int64

# 2.2 Extracting Mileage Units and Numeric Values

In [11]:
#creating a separate column for  mileage units
df_clean['mileage_unit']=df_clean['mileage'].str.extract(r'(kmpl|km/kg)',expand=False)

In [12]:
#creating a separate column for mileage numeric values
df_clean['mileage_value']=df_clean['mileage'].str.extract(r'(\d+(?:\.\d+)?)').astype(float)

In [13]:
#verifying the extracted columns
df_clean[['mileage','mileage_unit','mileage_value']].head(10)

,mileage,mileage_unit,mileage_value
0,23.4 kmpl,kmpl,23.40
1,21.14 kmpl,kmpl,21.14
2,17.7 kmpl,kmpl,17.70
3,23.0 kmpl,kmpl,23.00
4,16.1 kmpl,kmpl,16.10
5,20.14 kmpl,kmpl,20.14
6,17.3 km/kg,km/kg,17.30
7,16.1 kmpl,kmpl,16.10
8,23.59 kmpl,kmpl,23.59
9,20.0 kmpl,kmpl,20.00


In [14]:
#checking null values in the extracted columns 
df_clean[['mileage_value','mileage_unit']].isnull().sum()

mileage_value    208
mileage_unit     208
dtype: int64

# 3. Engine Cleaning

## 3.1 Exploring Engine Values and Units

In [15]:
#checking engine values
df_clean['engine'].head(10)

0    1248 CC
1    1498 CC
2    1497 CC
3    1396 CC
4    1298 CC
5    1197 CC
6    1061 CC
7     796 CC
8    1364 CC
9    1399 CC
Name: engine, dtype: str

In [16]:
#checking the units of engine
df_clean['engine'].str.extract(r'([A-Za-z]+)',expand=False).value_counts()

engine
CC    6718
Name: count, dtype: int64

## 3.2 Extracting Numeric Engine Values

In [17]:
#create a separate column for numeric engine values
df_clean['engine_value']=df_clean['engine'].str.extract(r'(\d+)').astype(float) 

In [18]:
#verifying the extracted engine values
df_clean[['engine','engine_value']].head(10)

,engine,engine_value
0,1248 CC,1248.0
1,1498 CC,1498.0
2,1497 CC,1497.0
3,1396 CC,1396.0
4,1298 CC,1298.0
5,1197 CC,1197.0
6,1061 CC,1061.0
7,796 CC,796.0
8,1364 CC,1364.0
9,1399 CC,1399.0


In [19]:
#checking for null values in the extracted column
df_clean[['engine', 'engine_value']].isnull().sum()


engine          208
engine_value    208
dtype: int64

# 4. Max Power Cleaning

## 4.1 Exploring Max Power Values and Units

In [20]:
#checking max power values
df_clean['max_power'].head(10)

0        74 bhp
1    103.52 bhp
2        78 bhp
3        90 bhp
4      88.2 bhp
5     81.86 bhp
6      57.5 bhp
7        37 bhp
8      67.1 bhp
9      68.1 bhp
Name: max_power, dtype: str

In [21]:
#checking if there are any other units in max power
df_clean['max_power'].str.extract(r'([A-Za-z]+)',expand=False).value_counts()

max_power
bhp    6718
Name: count, dtype: int64

## 4.2 Extracting Numeric Max Power Values

In [22]:
#creating a separate column for numeric values in max power
df_clean['max_power_value']=df_clean['max_power'].str.extract(r'(\d+(?:\.\d+)?)').astype(float)

In [23]:
#verifying the extracted column
df_clean[['max_power','max_power_value']].head(10)

,max_power,max_power_value
0,74 bhp,74.00
1,103.52 bhp,103.52
2,78 bhp,78.00
3,90 bhp,90.00
4,88.2 bhp,88.20
5,81.86 bhp,81.86
6,57.5 bhp,57.50
7,37 bhp,37.00
8,67.1 bhp,67.10
9,68.1 bhp,68.10


In [24]:
#checking missing values in extracted column
df_clean[['max_power','max_power_value']].isnull().sum()

max_power          205
max_power_value    206
dtype: int64

## 4.3 Handling Malformed and Invalid Values

In [25]:
#checking which value isn't converted into a numerical value
df_clean.loc[df_clean['max_power'].notnull()&df_clean['max_power_value'].isnull(),'max_power']

4933     bhp
Name: max_power, dtype: str

In [26]:
#replacing 'bhp' with no numeric value with NaN in original column to maintain consistency
df_clean.loc[df_clean['max_power'].str.strip()=='bhp','max_power']=np.nan

In [27]:
#checking missing values after correcting the invalid entry
df_clean[['max_power','max_power_value']].isnull().sum()

max_power          206
max_power_value    206
dtype: int64

# 5. Torque Cleaning

## 5.1 Exploring Torque Values and Units

In [28]:
#checking torque values
df_clean['torque'].head(10)

0              190Nm@ 2000rpm
1         250Nm@ 1500-2500rpm
2       12.7@ 2,700(kgm@ rpm)
3    22.4 kgm at 1750-2750rpm
4       11.5@ 4,500(kgm@ rpm)
5           113.75nm@ 4000rpm
6        7.8@ 4,500(kgm@ rpm)
7               59Nm@ 2500rpm
8         170Nm@ 1800-2400rpm
9              160Nm@ 2000rpm
Name: torque, dtype: str

## 5.2 Extracting Torque Units and Numeric Values

In [29]:
#creating a separate column for torque numeric values
df_clean['torque_value']=df_clean['torque'].str.extract(r'(\d+(?:\.\d+)?)').astype(float)

In [30]:
#checking missing values in extracted column
df_clean[['torque','torque_value']].isnull().sum()

torque          209
torque_value    209
dtype: int64

# 5.3 Handling Missing and Ambiguous Units 

In [31]:
#creating a column for units ignoring case differences
import re
df_clean['torque_unit']=df_clean['torque'].str.extract(r'(nm|kgm)',flags=re.IGNORECASE,expand=False).str.lower()

In [32]:
#count the occurrence of each unit,including missing values
df_clean['torque_unit'].value_counts(dropna=False)

torque_unit
nm     6227
kgm     480
NaN     219
Name: count, dtype: int64

In [33]:
#identifying the other values with missing units
df_clean.loc[df_clean['torque'].notnull()&df_clean['torque_unit'].isnull(),'torque']

140     250@ 1250-5000rpm
1676       510@ 1600-2400
2000      110(11.2)@ 4800
4373           210 / 1900
5572           210 / 1900
5785           210 / 1900
6418           210 / 1900
7154           210 / 1900
7296           210 / 1900
7532           210 / 1900
Name: torque, dtype: str

In [34]:
#checking the records of torque values with no units
df_clean.loc[df_clean['torque'].notnull()&df_clean['torque_unit'].isnull(),['name','fuel','engine','max_power','torque']]

,name,fuel,engine,max_power,torque
140,Skoda Superb LK 1.8 TSI AT,Petrol,1798 CC,177.46 bhp,250@ 1250-5000rpm
1676,Mercedes-Benz M-Class ML 350 4Matic,Diesel,2987 CC,165 bhp,510@ 1600-2400
2000,Honda Jazz Select Edition Active,Petrol,1198 CC,90 bhp,110(11.2)@ 4800
4373,Skoda Octavia Classic 1.9 TDI MT,Diesel,1896 CC,66 bhp,210 / 1900
5572,Skoda Octavia Classic 1.9 TDI MT,Diesel,1896 CC,66 bhp,210 / 1900
5785,Skoda Octavia Ambiente 1.9 TDI MT,Diesel,1896 CC,66 bhp,210 / 1900
6418,Skoda Octavia Ambiente 1.9 TDI MT,Diesel,1896 CC,66 bhp,210 / 1900
7154,Skoda Octavia Rider 1.9 AT TDI,Diesel,1896 CC,66 bhp,210 / 1900
7296,Skoda Octavia Ambiente 1.9 TDI,Diesel,1896 CC,66 bhp,210 / 1900
7532,Skoda Octavia Ambiente 1.9 TDI MT,Diesel,1896 CC,66 bhp,210 / 1900


# 6. Identifying and Handling Invalid Zero Values

In [35]:
#checking for zero values in four extracted numeric columns
df_clean[['mileage_value','engine_value','max_power_value','torque_value']].eq(0).sum()

mileage_value      15
engine_value        0
max_power_value     3
torque_value        0
dtype: int64

In [36]:
#checking the 0 mileage records
df_clean.loc[df_clean['mileage_value']==0]

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats,mileage_unit,mileage_value,engine_value,max_power_value,torque_value,torque_unit
644,Tata Indica Vista Aura Safire Anniversary Edition,2009,135000,28900,Petrol,Individual,Manual,Second Owner,0.0 kmpl,1172 CC,65 bhp,"9.8@ 3,000(kgm@ rpm)",5.0,kmpl,0.0,1172.0,65.00,9.8,kgm
785,Hyundai Santro Xing GL,2009,120000,90000,Petrol,Individual,Manual,Second Owner,0.0 kmpl,1086 CC,62 bhp,96.1Nm@ 3000rpm,5.0,kmpl,0.0,1086.0,62.00,96.1,nm
1649,Hyundai Santro Xing GL,2008,105000,128000,Petrol,Individual,Manual,First Owner,0.0 kmpl,1086 CC,62 bhp,96.1Nm@ 3000rpm,5.0,kmpl,0.0,1086.0,62.00,96.1,nm
1676,Mercedes-Benz M-Class ML 350 4Matic,2011,1700000,110000,Diesel,Individual,Automatic,Third Owner,0.0 kmpl,2987 CC,165 bhp,510@ 1600-2400,5.0,kmpl,0.0,2987.0,165.00,510.0,NaN
2137,Land Rover Freelander 2 TD4 HSE,2013,1650000,64788,Diesel,Dealer,Automatic,First Owner,0.0 kmpl,2179 CC,115 bhp,400 Nm /2000 rpm,5.0,kmpl,0.0,2179.0,115.00,400.0,nm
2366,Hyundai Santro Xing (Non-AC),2010,110000,80000,Petrol,Individual,Manual,Second Owner,0.0 kmpl,1086 CC,62.1 bhp,96.1Nm@ 3000rpm,5.0,kmpl,0.0,1086.0,62.10,96.1,nm
2725,Hyundai Santro Xing (Non-AC),2013,184000,15000,Petrol,Individual,Manual,First Owner,0.0 kmpl,1086 CC,62.1 bhp,96.1Nm@ 3000rpm,5.0,kmpl,0.0,1086.0,62.10,96.1,nm
5276,Hyundai Santro Xing GL,2008,175000,40000,Petrol,Individual,Manual,First Owner,0.0 kmpl,1086 CC,62 bhp,96.1Nm@ 3000rpm,5.0,kmpl,0.0,1086.0,62.00,96.1,nm
5843,Volkswagen Polo GT TSI BSIV,2014,574000,28080,Petrol,Dealer,Automatic,First Owner,0.0 kmpl,1197 CC,103.25 bhp,175nm@ 1500-4100rpm,5.0,kmpl,0.0,1197.0,103.25,175.0,nm
5846,Volkswagen Polo GT TSI BSIV,2014,575000,28100,Petrol,Dealer,Automatic,First Owner,0.0 kmpl,1197 CC,103.25 bhp,175nm@ 1500-4100rpm,5.0,kmpl,0.0,1197.0,103.25,175.0,nm


In [37]:
#replacing invalid zero mileage values with NaN
df_clean.loc[df_clean['mileage_value']==0,'mileage_value']=np.nan

In [38]:
#verifying the missing values
df_clean['mileage_value'].isnull().sum()

np.int64(223)

In [39]:
#checking 0 max power values
df_clean.loc[df_clean['max_power_value']==0]

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats,mileage_unit,mileage_value,engine_value,max_power_value,torque_value,torque_unit
575,Maruti Alto K10 LXI,2011,204999,97500,Petrol,Individual,Manual,First Owner,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
1442,Maruti Swift Dzire VDI Optional,2017,589000,41232,Diesel,Dealer,Manual,First Owner,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
2549,Tata Indica Vista Quadrajet LS,2012,240000,70000,Diesel,Individual,Manual,First Owner,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN


In [40]:
#replacing invalid zero max power values with NaN
df_clean.loc[df_clean['max_power_value']==0,'max_power_value']=np.nan

In [41]:
#verifying missing values
df_clean['max_power_value'].isnull().sum()

np.int64(209)

# 7. Missing Value Analysis 

In [42]:
#check the no.of missing values in the extracted numeric columns
df_clean[['mileage_value','engine_value','max_power_value','torque_value']].isnull().sum()

mileage_value      223
engine_value       208
max_power_value    209
torque_value       209
dtype: int64

In [43]:
#verifying missing values of all extracted columns
df_clean[['mileage_value','mileage_unit','engine_value','max_power_value','torque_value','torque_unit']].isnull().sum()

mileage_value      223
mileage_unit       208
engine_value       208
max_power_value    209
torque_value       209
torque_unit        219
dtype: int64

In [44]:
#percentage of missing values in four columns
(df_clean[['mileage_value','engine_value','max_power_value','torque_value']].isnull().mean()*100).round(2)

mileage_value      3.22
engine_value       3.00
max_power_value    3.02
torque_value       3.02
dtype: float64

In [45]:
#checking no.of missing values each row contains for these four columns
df_clean[['mileage_value','engine_value','max_power_value','torque_value']].isnull().sum(axis=1).value_counts().sort_index()

0    6702
1      15
2       1
4     208
Name: count, dtype: int64

In [46]:
#record which have 2 missing values
df_clean.loc[
    df_clean[['mileage_value', 'engine_value',
              'max_power_value', 'torque_value']].isnull().sum(axis=1)==2]

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats,mileage_unit,mileage_value,engine_value,max_power_value,torque_value,torque_unit
4933,Maruti Omni CNG,2000,80000,100000,CNG,Individual,Manual,Second Owner,10.9 km/kg,796 CC,NaN,NaN,8.0,km/kg,10.9,796.0,NaN,NaN,NaN


In [47]:
#records with 1 missing value
df_clean.loc[
    df_clean[['mileage_value', 'engine_value',
              'max_power_value', 'torque_value']].isnull().sum(axis=1)==1]

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats,mileage_unit,mileage_value,engine_value,max_power_value,torque_value,torque_unit
644,Tata Indica Vista Aura Safire Anniversary Edition,2009,135000,28900,Petrol,Individual,Manual,Second Owner,0.0 kmpl,1172 CC,65 bhp,"9.8@ 3,000(kgm@ rpm)",5.0,kmpl,NaN,1172.0,65.00,9.8,kgm
785,Hyundai Santro Xing GL,2009,120000,90000,Petrol,Individual,Manual,Second Owner,0.0 kmpl,1086 CC,62 bhp,96.1Nm@ 3000rpm,5.0,kmpl,NaN,1086.0,62.00,96.1,nm
1649,Hyundai Santro Xing GL,2008,105000,128000,Petrol,Individual,Manual,First Owner,0.0 kmpl,1086 CC,62 bhp,96.1Nm@ 3000rpm,5.0,kmpl,NaN,1086.0,62.00,96.1,nm
1676,Mercedes-Benz M-Class ML 350 4Matic,2011,1700000,110000,Diesel,Individual,Automatic,Third Owner,0.0 kmpl,2987 CC,165 bhp,510@ 1600-2400,5.0,kmpl,NaN,2987.0,165.00,510.0,NaN
2137,Land Rover Freelander 2 TD4 HSE,2013,1650000,64788,Diesel,Dealer,Automatic,First Owner,0.0 kmpl,2179 CC,115 bhp,400 Nm /2000 rpm,5.0,kmpl,NaN,2179.0,115.00,400.0,nm
2366,Hyundai Santro Xing (Non-AC),2010,110000,80000,Petrol,Individual,Manual,Second Owner,0.0 kmpl,1086 CC,62.1 bhp,96.1Nm@ 3000rpm,5.0,kmpl,NaN,1086.0,62.10,96.1,nm
2725,Hyundai Santro Xing (Non-AC),2013,184000,15000,Petrol,Individual,Manual,First Owner,0.0 kmpl,1086 CC,62.1 bhp,96.1Nm@ 3000rpm,5.0,kmpl,NaN,1086.0,62.10,96.1,nm
5276,Hyundai Santro Xing GL,2008,175000,40000,Petrol,Individual,Manual,First Owner,0.0 kmpl,1086 CC,62 bhp,96.1Nm@ 3000rpm,5.0,kmpl,NaN,1086.0,62.00,96.1,nm
5843,Volkswagen Polo GT TSI BSIV,2014,574000,28080,Petrol,Dealer,Automatic,First Owner,0.0 kmpl,1197 CC,103.25 bhp,175nm@ 1500-4100rpm,5.0,kmpl,NaN,1197.0,103.25,175.0,nm
5846,Volkswagen Polo GT TSI BSIV,2014,575000,28100,Petrol,Dealer,Automatic,First Owner,0.0 kmpl,1197 CC,103.25 bhp,175nm@ 1500-4100rpm,5.0,kmpl,NaN,1197.0,103.25,175.0,nm


In [48]:
#records with 4 missing values
df_clean.loc[
    df_clean[['mileage_value', 'engine_value',
              'max_power_value', 'torque_value']].isnull().all(axis=1)].head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats,mileage_unit,mileage_value,engine_value,max_power_value,torque_value,torque_unit
13,Maruti Swift 1.3 VXi,2007,200000,80000,Petrol,Individual,Manual,Second Owner,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31,Fiat Palio 1.2 ELX,2003,70000,50000,Petrol,Individual,Manual,Second Owner,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
78,Tata Indica DLS,2003,50000,70000,Diesel,Individual,Manual,First Owner,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
87,Maruti Swift VDI BSIV W ABS,2015,475000,78000,Diesel,Dealer,Manual,First Owner,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
119,Maruti Swift VDI BSIV,2010,300000,120000,Diesel,Individual,Manual,Second Owner,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [49]:
#missing values in seats column
df_clean['seats'].isnull().sum()

np.int64(208)

In [50]:
#checking if the same 208 rows missing for four columns also have missing seats
df_clean.loc[df_clean['mileage_value'].isnull()&df_clean['engine_value'].isnull()&df_clean['max_power_value'].isnull()&df_clean['torque_value'].isnull(),'seats'].isnull().sum()

np.int64(208)

# 8. Data Type and Consistency Checks

In [51]:
#datatypes of all the columns
df_clean.info()

<class 'pandas.DataFrame'>
Index: 6926 entries, 0 to 8125
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             6926 non-null   str    
 1   year             6926 non-null   int64  
 2   selling_price    6926 non-null   int64  
 3   km_driven        6926 non-null   int64  
 4   fuel             6926 non-null   str    
 5   seller_type      6926 non-null   str    
 6   transmission     6926 non-null   str    
 7   owner            6926 non-null   str    
 8   mileage          6718 non-null   str    
 9   engine           6718 non-null   str    
 10  max_power        6720 non-null   str    
 11  torque           6717 non-null   str    
 12  seats            6718 non-null   float64
 13  mileage_unit     6718 non-null   str    
 14  mileage_value    6703 non-null   float64
 15  engine_value     6718 non-null   float64
 16  max_power_value  6717 non-null   float64
 17  torque_value     6717 non-null

In [52]:
#checking car names
df_clean['name'].nunique()

2058

In [53]:
(df_clean['name']!=df_clean['name'].str.strip()).sum()

np.int64(0)

In [54]:
#checking the extracted numerical columns for negative values
(df_clean[['mileage_value','engine_value','max_power_value','torque_value']]<0).sum()

mileage_value      0
engine_value       0
max_power_value    0
torque_value       0
dtype: int64

# 9. Outlier Checks 

In [55]:
#checking ranges for numerical columns
df_clean[['mileage_value','engine_value','max_power_value']].describe()

,mileage_value,engine_value,max_power_value
count,6703.000000,6718.000000,6717.000000
mean,19.508869,1430.891337,87.766100
std,3.947453,493.493277,31.724555
min,9.000000,624.000000,32.800000
25%,16.800000,1197.000000,67.100000
50%,19.490000,1248.000000,81.830000
75%,22.540000,1498.000000,100.000000
max,42.000000,3604.000000,400.000000


In [56]:
#checking the record with maximum power value
df_clean.loc[df_clean['max_power_value']==df_clean['max_power_value'].max()]

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats,mileage_unit,mileage_value,engine_value,max_power_value,torque_value,torque_unit
170,Volvo XC90 T8 Excellence BSIV,2017,10000000,30000,Petrol,Individual,Automatic,First Owner,42.0 kmpl,1969 CC,400 bhp,640Nm@ 1740rpm,4.0,kmpl,42.0,1969.0,400.0,640.0,nm


In [57]:
#checking the records with maximum torque value
df_clean.loc[df_clean['torque_value']==df_clean['torque_value'].max()]

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats,mileage_unit,mileage_value,engine_value,max_power_value,torque_value,torque_unit
951,Maruti Zen D,2003,85000,120000,Diesel,Individual,Manual,Third Owner,20.8 kmpl,1527 CC,58 bhp,789Nm@ 2250rpm,5.0,kmpl,20.8,1527.0,58.0,789.0,nm
4720,Maruti Zen D,2002,120000,60000,Diesel,Individual,Manual,Second Owner,20.8 kmpl,1527 CC,58 bhp,789Nm@ 2250rpm,5.0,kmpl,20.8,1527.0,58.0,789.0,nm
5143,Maruti Zen D,2006,80000,110000,Diesel,Individual,Manual,Second Owner,20.8 kmpl,1527 CC,58 bhp,789Nm@ 2250rpm,5.0,kmpl,20.8,1527.0,58.0,789.0,nm


In [58]:
#checking the torque values for Maruti Zen D car model
df_clean.loc[
    df_clean['name'] == 'Maruti Zen D',
    ['year', 'engine', 'max_power', 'torque', 'torque_value']
]

,year,engine,max_power,torque,torque_value
951,2003,1527 CC,58 bhp,789Nm@ 2250rpm,789.0
4720,2002,1527 CC,58 bhp,789Nm@ 2250rpm,789.0
5143,2006,1527 CC,58 bhp,789Nm@ 2250rpm,789.0


## 9.1 Handling Suspicious Extreme Values

In [59]:
#replacing the suspiciously large torque values with missing values
df_clean.loc[df_clean['torque_value']==789,'torque_value']=np.nan

In [60]:
#verifying missing values of extracted torque numerical column
df_clean['torque_value'].isnull().sum()

np.int64(212)

# 10. Final Validation

In [61]:
#verifying null values for all columns
df_clean.isnull().sum()

name                 0
year                 0
selling_price        0
km_driven            0
fuel                 0
seller_type          0
transmission         0
owner                0
mileage            208
engine             208
max_power          206
torque             209
seats              208
mileage_unit       208
mileage_value      223
engine_value       208
max_power_value    209
torque_value       212
torque_unit        219
dtype: int64

In [62]:
#verifying duplicates for all columns
df_clean.duplicated().sum()

np.int64(0)

In [63]:
#verifying final shape 
df_clean.shape

(6926, 19)

# 11. Summary of Data Cleaning

- Removed Duplicate records
- Extracted numeric values and units for mixed data type columns.
- Identified and handled invalid, malformed and suspiciously extreme values with NaN.
- Analysed missing values and retained them.
- Checked the consistency of columns and verified that no duplicate records remained.

In [64]:
import os
os.makedirs('../data/processed', exist_ok=True)
df_clean.to_csv('../data/processed/used_cars_cleaned.csv',index=False)

In [65]:
pd.read_csv('../data/processed/used_cars_cleaned.csv').shape

(6926, 19)

In [66]:
df_saved = pd.read_csv('../data/processed/used_cars_cleaned.csv')

df_saved.isnull().sum()

name                 0
year                 0
selling_price        0
km_driven            0
fuel                 0
seller_type          0
transmission         0
owner                0
mileage            208
engine             208
max_power          206
torque             209
seats              208
mileage_unit       208
mileage_value      223
engine_value       208
max_power_value    209
torque_value       212
torque_unit        219
dtype: int64